In [ ]:
# ==============================================================================
# 1. SETUP & DEPENDENCIES
# ==============================================================================
import os, gc, torch, warnings
import pandas as pd
import numpy as np
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from datasets import load_dataset
from sacrebleu.metrics import BLEU, CHRF, TER
from bert_score import score as bert_score_fn
from nltk.translate.meteor_score import meteor_score
from nltk.tokenize import word_tokenize
import nltk

warnings.filterwarnings("ignore")
nltk.download(['wordnet', 'omw-1.4', 'punkt'], quiet=True)

# Install IndicTransToolkit if missing
try:
    from IndicTransToolkit import IndicProcessor
except ImportError:
    import subprocess
    subprocess.check_call(["pip", "install", "git+https://github.com/VarunGumma/IndicTransToolkit.git"])
    from IndicTransToolkit import IndicProcessor

# ==============================================================================
# 2. CONFIGURATION
# ==============================================================================
CONFIG = {
    "target_lang": "telugu", # options: "hindi", "telugu", "tamil"
    "lang_map": {"hindi": "hin_Deva", "telugu": "tel_Telu", "tamil": "tam_Taml"},
    "device": "cuda" if torch.cuda.is_available() else "cpu",
    "batch_size": 16,
    "max_length": 256,
    "thresholds": {
        "chrf": 30.0,
        "bleu": 20.0,
        "meteor": 50.0,
        "ter": 60.0,      # Keep if lower than this
        "bertscore": 85.0 # F1 Score percentage
    }
}

# ==============================================================================
# 3. PIPELINE FUNCTIONS
# ==============================================================================

def get_model(direction="en-indic"):
    """Load specific model for forward or backward translation."""
    model_name = "ai4bharat/indictrans2-en-indic-1B" if direction == "en-indic" else "prajdabre/rotary-indictrans2-indic-en-dist-200M"
    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True, use_fast=False)
    model = AutoModelForSeq2SeqLM.from_pretrained(
        model_name,
        trust_remote_code=True,
        torch_dtype=torch.float16 if CONFIG["device"] == "cuda" else torch.float32,
        device_map="auto" if CONFIG["device"] == "cuda" else None
    ).eval()
    return tokenizer, model

def translate(texts, tokenizer, model, src_lang, tgt_lang, is_back_trans=False):
    """Batch translation logic."""
    ip = IndicProcessor(inference=True)
    results = []

    for i in tqdm(range(0, len(texts), CONFIG["batch_size"]), desc=f"Translating ({src_lang}->{tgt_lang})"):
        batch = [str(t) for t in texts[i : i + CONFIG["batch_size"]]]

        if is_back_trans:
            # Back-translator uses prefix-based prompt logic for distilled models
            inputs = tokenizer([f"{src_lang} {tgt_lang} {s}" for s in batch],
                               padding=True, truncation=True, max_length=CONFIG["max_length"], return_tensors="pt").to(CONFIG["device"])
        else:
            # Forward translator uses the IndicProcessor
            processed = ip.preprocess_batch(batch, src_lang=src_lang, tgt_lang=tgt_lang)
            inputs = tokenizer(processed, padding=True, truncation=True, max_length=CONFIG["max_length"], return_tensors="pt").to(CONFIG["device"])

        with torch.no_grad():
            outputs = model.generate(**inputs, max_length=CONFIG["max_length"], use_cache=False)

        decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
        if not is_back_trans:
            decoded = ip.postprocess_batch(decoded, lang=tgt_lang)
        results.extend(decoded)

    return results

def compute_quality_metrics(originals, back_translations):
    """Compute BLEU, chrF++, TER, METEOR, and BERTScore."""
    # Sentence-level tokenized metrics
    bleu = BLEU(); chrf = CHRF(word_order=2); ter = TER()
    metrics = []

    for orig, back in tqdm(zip(originals, back_translations), total=len(originals), desc="Computing basic metrics"):
        metrics.append({
            "chrf_score": round(chrf.sentence_score(back, [orig]).score, 2),
            "bleu_score": round(bleu.sentence_score(back, [orig]).score, 2),
            "ter_score": round(ter.sentence_score(back, [orig]).score, 2),
            "meteor_score": round(meteor_score([word_tokenize(orig.lower())], word_tokenize(back.lower())) * 100, 2)
        })

    # Batch BERTScore (much faster than sentence-level)
    print("Computing BERTScore...")
    _, _, f1 = bert_score_fn(back_translations, originals, lang="en", device=CONFIG["device"], batch_size=CONFIG["batch_size"])

    df_metrics = pd.DataFrame(metrics)
    df_metrics["bertscore_f1"] = (f1.cpu().numpy() * 100).round(2)
    return df_metrics

# ==============================================================================
# 4. EXECUTION FLOW
# ==============================================================================

# Step 1: Data Loading
print(f"Loading dataset...")
ds = load_dataset("codesagar/malicious-llm-prompts-v4")
df = pd.concat([pd.DataFrame(ds["train"]), pd.DataFrame(ds["test"])], ignore_index=True)
# df = df.head(100) # Uncomment for quick testing
english_originals = df["prompt"].tolist()

# Step 2: Forward Translation (EN -> Indic)
tok, mod = get_model("en-indic")
indic_texts = translate(english_originals, tok, mod, "eng_Latn", CONFIG["lang_map"][CONFIG["target_lang"]])
del tok, mod; gc.collect(); torch.cuda.empty_cache() # Essential memory cleanup

# Step 3: Back Translation (Indic -> EN)
tok, mod = get_model("indic-en")
back_translations = translate(indic_texts, tok, mod, CONFIG["lang_map"][CONFIG["target_lang"]], "eng_Latn", is_back_trans=True)
del tok, mod; gc.collect(); torch.cuda.empty_cache()

# Step 4: Metrics Computation
metrics_df = compute_quality_metrics(english_originals, back_translations)

# Step 5: Consolidation & Filtering
final_df = pd.concat([
    df.reset_index(drop=True),
    pd.Series(indic_texts, name="translated_prompt"),
    pd.Series(back_translations, name="back_translated_prompt"),
    metrics_df
], axis=1)

mask = (
    (final_df["chrf_score"] >= CONFIG["thresholds"]["chrf"]) &
    (final_df["bleu_score"] >= CONFIG["thresholds"]["bleu"]) &
    (final_df["meteor_score"] >= CONFIG["thresholds"]["meteor"]) &
    (final_df["bertscore_f1"] >= CONFIG["thresholds"]["bertscore"]) &
    (final_df["ter_score"] <= CONFIG["thresholds"]["ter"])
)

filtered_df = final_df[mask]

# Step 6: Output
output_file = f"filtered_{CONFIG['target_lang']}_dataset.xlsx"
filtered_df.to_excel(output_file, index=False)

print(f"\nPipeline Complete.")
print(f"Total processed: {len(final_df)} | Retained after filtering: {len(filtered_df)}")
print(f"Result saved to: {output_file}")